# Model Comparison
Comparing 4 models across repository branches on `data.csv` using 5-Fold Stratified Cross-Validation:
- LinearSVC with Character Subword TF-IDF (Notebook model)
- Logistic Regression with Word TF-IDF (`main` branch)
- Multinomial Naive Bayes with Word TF-IDF (`main` & `multinomial-naive-bayes` branch)
- KNN (k=5) with Word TF-IDF (`esther-knn` branch)


### 1. Setup Data

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

# Load cleaned dataset
df = pd.read_csv("data.csv")
X = np.array(df['text'].tolist(), dtype=object)
y = np.array(df['label'].tolist(), dtype=object)

print(f"Loaded: {len(df)} total messages (Ham: {sum(y=='ham')}, Spam: {sum(y=='spam')})")

Loaded: 19582 total messages (Ham: 10586, Spam: 8996)


### 2. 5-Fold Stratified CV
LinearSVC has higest Accuracy (98.84%) and F1 Score (95.25%), missing 51 spam messages compared to 120 (Logistic), 233 (Naive Bayes), and 300 (KNN).

In [3]:
models = {
    "LinearSVC (Char Subword)": make_pipeline(
        TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True),
        LinearSVC(C=1.0, random_state=42, dual='auto')
    ),
    "Logistic Regression (Word)": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=200)
    ),
    "Multinomial Naive Bayes (Word)": make_pipeline(
        TfidfVectorizer(),
        MultinomialNB()
    ),
    "KNN (Word, k=5)": make_pipeline(
        TfidfVectorizer(),
        KNeighborsClassifier(n_neighbors=5)
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
oof_predictions = {}

for name, pipe in models.items():
    all_true, all_pred = [], []
    for train_idx, test_idx in cv.split(X, y):
        pipe.fit(X[train_idx], y[train_idx])
        all_true.extend(y[test_idx])
        all_pred.extend(pipe.predict(X[test_idx]))
        
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    oof_predictions[name] = (all_true, all_pred)
    
    acc = accuracy_score(all_true, all_pred)
    prec = precision_score(all_true, all_pred, pos_label='spam')
    rec = recall_score(all_true, all_pred, pos_label='spam')
    f1 = f1_score(all_true, all_pred, pos_label='spam')
    tn, fp, fn, tp = confusion_matrix(all_true, all_pred, labels=['ham', 'spam']).ravel()
    
    results.append({
        'Model': name,
        'Accuracy': f"{acc * 100:.2f}%",
        'Precision': f"{prec * 100:.2f}%",
        'Recall': f"{rec * 100:.2f}%",
        'F1 Score': f"{f1 * 100:.2f}%",
        'False Positives (FP)': fp,
        'False Negatives (FN)': fn
    })

results_df = pd.DataFrame(results)
display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score,False Positives (FP),False Negatives (FN)
0,LinearSVC (Char Subword),97.74%,97.50%,97.58%,97.54%,225,218
1,Logistic Regression (Word),95.65%,95.47%,95.04%,95.25%,406,446
2,Multinomial Naive Bayes (Word),91.67%,99.04%,82.67%,90.12%,72,1559
3,"KNN (Word, k=5)",90.04%,88.42%,90.12%,89.26%,1062,889


In [4]:
weight_files = {
    "LinearSVC (Char Subword)": "model_linearsvc.joblib",
    "Logistic Regression (Word)": "model_logistic_regression.joblib",
    "Multinomial Naive Bayes (Word)": "model_naive_bayes.joblib",
    "KNN (Word, k=5)": "model_knn.joblib"
}

# Only train if joblib file does not exist, otherwise load
loaded_models = {}
for name, pipe in models.items():
    file = weight_files[name]
    if os.path.exists(file):
        loaded_models[name] = joblib.load(file)
    else:
        pipe.fit(X, y)
        joblib.dump(pipe, file)
        loaded_models[name] = pipe

testcases = [
    ("Ham Edge Case", "Congratulation daughter, you graduated today!", "ham"),
    ("Everyday Message", "Are you free for lunch at 12:30 tomorrow?", "ham"),
    ("Trigger-Word Trap", "Team fixed the prize bug in the mobile game today", "ham"),
    ("Typo Obfuscation", "URG3NT: Ur b@nk acct is locked. Call [phone] to unlock", "spam"),
    ("Spear Phishing", "In a meeting right now, please buy 5 Apple gift cards for the client", "spam")
]

test_rows = []
for desc, text, truth in testcases:
    row = {"Scenario": desc, "Expected": truth.upper()}
    for name, pipe in loaded_models.items():
        pred = pipe.predict([text])[0]
        row[name] = "PASS" if pred == truth else f"FAIL ({pred.upper()})"
    test_rows.append(row)

display(pd.DataFrame(test_rows))

,Scenario,Expected,LinearSVC (Char Subword),Logistic Regression (Word),Multinomial Naive Bayes (Word),"KNN (Word, k=5)"
0,Ham Edge Case,HAM,PASS,FAIL (SPAM),FAIL (SPAM),FAIL (SPAM)
1,Everyday Message,HAM,PASS,PASS,PASS,PASS
2,Trigger-Word Trap,HAM,PASS,PASS,PASS,PASS
3,Typo Obfuscation,SPAM,PASS,PASS,PASS,FAIL (HAM)
4,Spear Phishing,SPAM,PASS,PASS,PASS,PASS


### 3. Testcases Analysis
- LinearSVC and Logistic Regression pass the typo obfuscation test (`URG3NT: Ur b@nk`).
- Word-level Naive Bayes and KNN fail on typos because misspelled words are not in their dictionaries.
- All models fail on spear phishing because it uses normal workplace vocabulary without any spam keywords.
(this one is hard for even human so maybe it's fine)